In [ ]:
from dataclasses import dataclass
from typing import Tuple
import jax
import jax.numpy as jnp
from jax import jit, lax
from torch.distributions.normal import Normal
import contextlib
import io
import os
import sys
from types import SimpleNamespace

sys.path.append('../..')
path=os.path.abspath(os.path.join(os.getcwd(), "../../outputs/kuramoto/ppo"))

import gc

from jax.random import PRNGKey
import distrax
import numpy as np
from functools import partial
from IPython.display import clear_output

import torch
import torch.nn as nn
from tqdm import tqdm
from adabelief_pytorch import AdaBelief
from kuramoto_env import activate_state, init_sim_params, heun_step, SimState, count_reward_batch, criticalK, order_param
from core.policy_ppo import ActorCritic, compute_gae, ppo_update, find_params
from configs.kuramoto_ppo_cfg import cfg
is_cuda = torch.cuda.is_available()
device = 'cuda' if is_cuda else 'cpu'

In [ ]:
#download the adjacency matrix from cache
%store -r adj_matrix

A = adj_matrix
dim=A.shape[0]
# compute critical K and Laplacian
Kcrit, L = criticalK(A, dim, 5.0)
K = 0.01 * Kcrit

print(K)

In [ ]:
def train_ppo_run(a_layers, c_layers, config, A_matrix, L_matrix):
    """
    Executes a full PPO training suite with explicit 'dones' tracking.
    """
    with contextlib.redirect_stdout(io.StringIO()): 
        model = ActorCritic(3*dim + 1, dim, a_layers, c_layers, config.omniscent).to(device)
        optimizer = AdaBelief(model.parameters(), lr=config.learning_rate, rectify=True)
        scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.999)
    
    metrics = {"cost": [], "order_p": [], "ema": []}
    ema_loss, best_score, key = 0.0, float('inf'), PRNGKey(0)

    pbar = tqdm(range(config.epochs_per_config), desc=f"PPO A{a_layers} C{c_layers}")
    
    for epoch in pbar:
        # Reset Environment
        key, sub1, sub2 = jax.random.split(key, 3)
        state = activate_state(sub1, config.batch_size, dim)
        sim_params = init_sim_params(sub2, A_matrix, K, config.batch_size)
        
        # Explicit Rollout Buffers
        rollout = {k: [] for k in ["obs", "acts", "log_p", "vals", "rews", "dones"]}
        energy_accum, reward_accum = 0, 0

        # --- 1. Interaction Loop ---
        for t in range(config.steps_per_trajectory):
            # State Observation
            obs_jax = jnp.concatenate([
                jnp.sin(state.N), jnp.cos(state.N), 
                jnp.expand_dims(state.t * (1.0/config.steps_per_trajectory), 1), 
                sim_params.freqs
            ], axis=1)
            obs_torch = torch.as_tensor(np.array(obs_jax), device=device)

            with torch.no_grad():
                mean, std, value = model(obs_torch)
                dist = Normal(mean, config.noise_rms * std * (1.0/config.steps_per_trajectory))
                action = dist.sample()
                log_p = dist.log_prob(action).sum(dim=-1)

            # Step Environment
            action_np = action.cpu().numpy()
            n_next = heun_step(state.N, jnp.array(action_np), sim_params, 1.0/config.steps_per_trajectory)
            reward, cost_r1, _, penalty = count_reward_batch(
                state.N, n_next, jnp.array(action_np), sim_params.A, config.reward_weight_r, 1.0/config.steps_per_trajectory
            )
            
            # Determine Done Signal
            done = (t == config.steps_per_trajectory - 1)
            
            # Store Step Data
            rollout["obs"].append(obs_torch)
            rollout["acts"].append(action)
            rollout["log_p"].append(log_p)
            rollout["vals"].append(value)
            rollout["rews"].append(torch.as_tensor(np.array(reward), dtype=torch.float32, device=device))
            rollout["dones"].append(torch.as_tensor(float(done), dtype=torch.float32, device=device))
            
            state = SimState(N=n_next, t=state.t + 1)
            energy_accum += np.array(penalty)
            reward_accum += np.mean(reward)

        # --- 2. Post-Trajectory Calculations ---
        final_order = order_param(state.N, L_matrix)
        total_epoch_loss = np.mean(cost_r1 + energy_accum)
        ema_loss = total_epoch_loss if epoch == 0 else (ema_loss * config.alpha_ema + total_epoch_loss * (1 - config.alpha_ema))

        # --- 3. GAE and PPO Update ---
        # Convert lists to tensors
        t_rollout = {k: torch.stack(v) for k, v in rollout.items()}
        
        with torch.no_grad():
            # Get next value for bootstrapping
            obs_final_jax = jnp.concatenate([jnp.sin(state.N), jnp.cos(state.N), 
                                            jnp.expand_dims(state.t * (1.0/config.steps_per_trajectory), 1), 
                                            sim_params.freqs], axis=1)
            obs_final_torch = torch.as_tensor(np.array(obs_final_jax), device=device)
            _, _, next_val = model(obs_final_torch)

        # Compute Advantages using restored Dones
        advantages, returns = compute_gae(
            t_rollout["rews"], t_rollout["vals"], next_val, 
            t_rollout["dones"], config.gamma, config.gae_lambda
        )
        
        ppo_update(
            model, optimizer, scheduler, t_rollout["obs"], t_rollout["acts"], 
            t_rollout["log_p"], advantages, returns, config.clip_epsilon, config.batch_size
        )

        # --- 4. Logging & Cleanup ---
        if ema_loss < best_score:
            best_score = ema_loss
            torch.save(model.state_dict(), path +"/" + str(a_layers)+str(c_layers)+".pth")
     
        pbar.set_postfix({"EMA": f"{ema_loss:.2e}", "OrderP": f"{np.mean(final_order):.3f}"})
        
        if epoch % 500 == 0:
            torch.cuda.empty_cache()
            
    return metrics

In [ ]:
results_registry = []

for a_l in range(5):
    for c_l in range(5):
        print(f"\n--- Starting Architecture: Actor {a_l} | Critic {c_l} ---")
        run_data = train_ppo_run(a_l, c_l, cfg, A, L)
        results_registry.append(run_data)
        
        # Final cleanup for the config
        gc.collect()
        torch.cuda.empty_cache()